In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import Lambda
from sklearn.model_selection import train_test_split

### Initialization backbone (our package) Example

In [2]:
import backbone as bb

In [3]:
#This part will contain all constant variables (which don't change)
#Add other values in the future for readability
HEIGHT = 256
WIDTH = 256

In [4]:
#file_path_trainlabels = path name to train labels (img_name and label) of the .csv file
#file_path_trainimages = path name to train images
#train_labels = pandas dataframe of the .csv file
file_path_trainlabels, file_path_trainimages, train_labels = bb.init_data()

In [5]:
#x_trainname = array of 'img_name' elements for the train dataset (after splitting)
#x_valname = array of 'img_name' elements for the validation dataset (after splitting)
#y_trainlabel = numpy array of train labels matching the images with name of x_trainname
#y_vallabel = numpy array of validation labels matching the images with name of x_valname
x_trainname, x_valname, y_trainlabel, y_vallabel = bb.make_split(0.2)

In [6]:
#x_trainimgs = numpy array of matrices contraining all images of the training set
#x_valimgs = numpy array of matrices contraining all images of the validation set
x_trainimgs,x_valimgs = bb.make_image_sets(HEIGHT,WIDTH,x_trainname,x_valname)

In [7]:
#file_path_testimages = path name to test labels (img_name and label) of the .csv file
#file_path_testlabels = path name to test images
#test_labels = pandas dataframe of the .csv file of the 'sample.csv'
file_path_testimages,file_path_testlabels,test_labels = bb.get_test_paths()

### Exploration

In [8]:
#image = plt.imread(file_path_trainimages + 'train_1.jpg')
#plt.imshow(image)
#train_labels.head()
#train_labels['label'].value_counts() 
#image
#unique_numbers = set(train_labels['label'])
#unique_numbers
#train_data = tf.data.Dataset.from_tensor_slices((x_trainimgs, y_trainlabel))
#valid_data = tf.data.Dataset.from_tensor_slices((x_valimgs, y_vallabel))

### Making initial test network

In [9]:
test_model = models.Sequential()
test_model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(HEIGHT, WIDTH,3)))
test_model.add(layers.MaxPooling2D((2, 2)))
test_model.add(layers.Conv2D(64, (3, 3), activation='relu'))
test_model.add(layers.Flatten())
test_model.add(layers.Dense(81))
test_model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 254, 254, 32)      896       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 127, 127, 32)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 125, 125, 64)      18496     
_________________________________________________________________
flatten (Flatten)            (None, 1000000)           0         
_________________________________________________________________
dense (Dense)                (None, 81)                81000081  
Total params: 81,019,473
Trainable params: 81,019,473
Non-trainable params: 0
_________________________________________________________________


In [10]:
test_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [11]:
fitted_model = test_model.fit(x_trainimgs,y_trainlabel, epochs=1,
                    validation_data=(x_valimgs, y_vallabel))

766/766 [==============================] - 655s 855ms/step - loss: 82.3282 - accuracy: 0.0176 - val_loss: 4.3729 - val_accuracy: 0.0173


In [19]:
def get_scores(test_model, height, width, file_path_testimages, test_labels, ipm="nearest"):
    scores = []

    for i in test_labels['img_name']:
        img = keras.preprocessing.image.load_img(
        file_path_testimages + i, target_size=(height, width), interpolation = ipm
        )
        img_array = keras.preprocessing.image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)
        predictions = test_model.predict(img_array)
        score = tf.nn.softmax(predictions[0])
        scores.append(np.argmax(score))
    
    return scores

In [20]:
#This function makes an array of scores per image
scores = get_scores(test_model, HEIGHT, WIDTH, file_path_testimages, test_labels)

### Output Example

In [29]:
#This function takes all the labels of test images and the matching scores and makes a 
#.csv file from it
data_export(test_labels['img_name'],scores)